In [57]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import random
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch.optim as optim
from tqdm import tqdm



In [58]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
# device = torch.device("cuda")
device = torch.device("cpu")

# Przygotowanie i funkcjie pomocnicze

## Wczytywanie danych

In [ ]:
def load_data(data_folder, transform=None):
    if transform is None:
        transform = transforms.Compose(
            [transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

    dataset = torchvision.datasets.ImageFolder(root=data_folder, transform=transform)
    return dataset

def split_train_val_dataset(dataset, train_ratio=0.8):
    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = total_size - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    return train_dataset, val_dataset

def train_test_split(X, y, test_size=0.2):
    total_size = len(X)
    test_size = int(test_size * total_size)
    indices = list(range(total_size))
    random.shuffle(indices)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]
    X_train = [X[i] for i in train_indices]
    y_train = [y[i] for i in train_indices]
    X_test = [X[i] for i in test_indices]
    y_test = [y[i] for i in test_indices]
    return X_train, X_test, y_train, y_test

# nieuzywane
def number_of_classes(dataset):
    return len(dataset.classes)

def define_dataloaders(train_dataset, val_dataset, batch_size=64):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

## Metryki

In [60]:
# def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
#     assert len(predictions) == len(targets)
#     accuracies = []
#     for i in range(n_classes):
#         accuracies.append((predictions[targets == i] == i).sum() / (targets == i).sum())
#     return np.mean(accuracies)

def get_accuracy(model, data, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.shape[0]
            correct += int((predicted == labels).sum())
    return correct / total



## Sieć

In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes=50, hidden_size=25):
        super().__init__()
        ## Warstwa konwolucyjna
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1, padding=0)
        ## Warstwa max pooling
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2)
        # !!!
        self.fc1 = nn.LazyLinear(hidden_size)
        # self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        # !!!
        x = torch.flatten(x, start_dim=1) # flatten all dimensions except batch
        # print(x.size())
        # !!!
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x
    
# skopiowane z lab6:
class LongNet(nn.Module):
	def __init__(self, num_classes=50, hidden_size=[5, 10]):
		super().__init__()
		self.n_convs = 10
		n_channels = 10

		self.conv1 = nn.Conv2d(in_channels=3, out_channels=n_channels, kernel_size=5, stride=1, padding=0)
		self.conv2 = nn.Conv2d(in_channels=n_channels, out_channels=n_channels, kernel_size=5, stride=1, padding=0)
		self.convs = nn.ModuleList(
			[nn.Conv2d(in_channels=n_channels * 2, out_channels=n_channels, kernel_size=5, stride=1, padding=2) for _ in range(self.n_convs)]
		)
		self.bns = nn.ModuleList([nn.BatchNorm2d(n_channels) for _ in range(self.n_convs)])
	
		self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

		self.fc1 = nn.LazyLinear(hidden_size[0])
		self.fc2 = nn.Linear(hidden_size[0], hidden_size[1])
		self.fc3 = nn.Linear(hidden_size[1], num_classes)

	def forward(self, x):
		x = F.relu(self.conv1(x))
		res = F.relu(self.conv2(x))
		x = res
		for i in range(self.n_convs):
			x = torch.cat([x, res], dim=1)
			x = F.relu(self.convs[i](x))
			x = self.bns[i](x)
		x = self.pool1(x)
		x = torch.flatten(x, 1)
		x = F.relu(self.fc1(x))
		x = F.relu(self.fc2(x))
		x = self.fc3(x)
		return x

## Funckja treningowa

In [70]:
def train_model(net, train_loader, val_loader, criterion, optimizer, eval_func, device, num_epochs=5, verbose=True):
    train_eval_hist = []
    val_eval_hist = []
    loss_hist = []

    rng = tqdm(range(num_epochs)) if verbose else range(num_epochs)

    for epoch in rng:
        net.train()
        running_loss = 0.0

        for data in train_loader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / 2000
        loss_hist.append(epoch_loss)
        print(epoch_loss)

        if not verbose:
            print('[%d/%d] loss: %.3f' % (epoch + 1, num_epochs, epoch_loss))
        else:
            rng.set_postfix(loss=f"{epoch_loss:.3f}")

        train_eval_hist.append(eval_func(net, train_loader, device))
        val_eval_hist.append(eval_func(net, val_loader, device))

    if not verbose:
        print('Finished Training')

    return loss_hist, train_eval_hist, val_eval_hist

# Trening i ...

In [ ]:
dataset = load_data("train/")
dataset_train, dataset_val = split_train_val_dataset(dataset)
train_loader, val_loader = define_dataloaders(dataset_train, dataset_val, batch_size=64)

labels = dataset.classes
data_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4) 

# coś tam gościu pisał o tym poniej na teams, ale nwm
# X_train, X_val, y_train, y_val = train_test_split(dataset_train, dataset_val)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled   = scaler.transform(X_val)
# X_test_scaled  = scaler.transform(X_test)

In [ ]:
set_seed(42)

criterion = nn.CrossEntropyLoss()
net = Net().to(device)
optimizer = optim.Adam(net.parameters(), lr=0.001)

# zakomentowane, bo trening trwa długo
# loss_hist, train_eval_hist, val_eval_hist = train_model(net, train_loader, val_loader, criterion, optimizer, get_accuracy, device, 5) 

100%|██████████| 5/5 [11:50<00:00, 142.06s/it, loss=1.466]


In [53]:
torch.save(net.state_dict(), "model_v1.tar")

images, img_labels = next(iter(val_loader))
images = images[:6]

output = net(images.to(device))
_, preds = torch.max(output, 1)

print("True labels: ", [labels[i] for i in img_labels[:6]])
print("Predicted labels: ", [labels[i] for i in preds])

final_acc = get_accuracy(net, data_loader, device)
print(f"Final accuracy: {final_acc}") 

True labels:  ['bird', 'fungus', 'camera', 'turtle', 'bridge', 'crab']
Predicted labels:  ['nest', 'spice', 'spice', 'fish', 'memorial', 'pizza']
Final accuracy: 0.2874981536398859


In [74]:
print(loss_hist, train_eval_hist, val_eval_hist)

[1.8236897305250168, 1.6341949923038483, 1.5525662722587585, 1.5024222233295441, 1.4658534290790557] [0.18235143733666628, 0.23845301670264743, 0.25306783320077264, 0.27028178616066356, 0.2904641518009317] [0.17695847298755893, 0.22933590865193432, 0.24103845935351928, 0.2534227120377208, 0.2756348349713117]


### 2

In [ ]:
# dataset = load_data("train/")
# dataset_train, dataset_val = split_train_val_dataset(dataset)
# train_loader, val_loader = define_dataloaders(dataset_train, dataset_val, batch_size=64)

# labels = dataset.classes
# data_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4) 

In [ ]:
# set_seed(SEED)

# criterion = nn.CrossEntropyLoss()
# long_net = LongNet().to(device)
# optimizer = optim.Adam(long_net.parameters(), lr=0.001)

# loss_hist, train_eval_hist, val_eval_hist = train_model(long_net, train_loader, val_loader, criterion, optimizer, get_accuracy, device, 1)

# torch.save(long_net.state_dict(), "model_v2_long.tar")

# # !!! COŚ DŁUGO SIE ROBI !!!

In [ ]:

# images, img_labels = next(iter(val_loader))
# images = images[:6]

# output = net(images.to(device))
# _, preds = torch.max(output, 1)

# print("True labels: ", [labels[i] for i in img_labels[:6]])
# print("Predicted labels: ", [labels[i] for i in preds])

# final_acc = get_accuracy(net, data_loader, device)
# print(f"Final accuracy: {final_acc}") 

# Rest

In [ ]:
# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

In [ ]:
# dataset = torchvision.datasets.ImageFolder(
#     root="train/",
#     transform=transform
# )

In [ ]:
# class_to_idx = dataset.class_to_idx
# idx_to_class = {v: k for k, v in class_to_idx.items()}

# print(class_to_idx)

{'acoustic': 0, 'antenna': 1, 'bacteria': 2, 'battery': 3, 'bean': 4, 'beetle': 5, 'bicycle': 6, 'birch': 7, 'bird': 8, 'bomb': 9, 'bread': 10, 'bridge': 11, 'camera': 12, 'carbon': 13, 'cat': 14, 'corn': 15, 'crab': 16, 'crocodilian': 17, 'echinoderm': 18, 'egg': 19, 'elephant': 20, 'fish': 21, 'flower': 22, 'frog': 23, 'fungus': 24, 'gauge': 25, 'hammer': 26, 'icecream': 27, 'kangaroo': 28, 'memorial': 29, 'monkey': 30, 'motor': 31, 'nest': 32, 'palm': 33, 'pizza': 34, 'pot': 35, 'printer': 36}


In [ ]:
# number_of_classes = len(class_to_idx)

In [ ]:
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size

# train_data, val_data = random_split(dataset, [train_size, val_size])

# train_loader = DataLoader(train_data, batch_size=32, shuffle=True, drop_last=True)
# val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

In [ ]:
# for images, labels in train_loader:
#     print(images.shape)  # e.g. [32, 3, 224, 224]
#     print(labels.shape)  # e.g. [32]
#     break

torch.Size([32, 3, 64, 64])
torch.Size([32])


In [ ]:
# all_samples = torch.stack([sample[0] for sample in dataset])
# print(f"Data norm: mean={all_samples.mean():.4f} | std={all_samples.std():.4f}")
# print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

In [ ]:
# def get_weights(weight_classes : bool = True):
#     if not weight_classes:
#         return torch.ones((1))
#     # targets = list of class indices for each image
#     counts = Counter(dataset.targets)

#     # Map back to class names
#     class_counts = {dataset.classes[i]: counts[i] for i in range(len(dataset.classes))}

#     min_class = min(class_counts, key=class_counts.get)
#     max_class = max(class_counts, key=class_counts.get)

#     ## bread ma 1108, carbon ma 503, reszta po 1800

#     weights = torch.tensor(list(class_counts.values()), dtype =float)
#     weights = torch.reciprocal(weights)

#     return weights

In [ ]:
# get_weights()

tensor([0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0009, 0.0006, 0.0006, 0.0020, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006, 0.0006,
        0.0006], dtype=torch.float64)

In [ ]:
# class Net(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ## Warstwa konwolucyjna
#         self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1, padding=0)
#         ## Warstwa max pooling
#         self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.conv2 = nn.Conv2d(6, 16, 5)
#         self.pool2 = nn.MaxPool2d(2)
#         # !!!
#         self.fc1 = nn.LazyLinear(120)
#         # self.fc1 = nn.Linear(16 * 5 * 5, 120)
#         self.fc2 = nn.Linear(120, 84)
#         self.fc3 = nn.Linear(84, number_of_classes)

#     def forward(self, x):
#         x = self.pool1(F.relu(self.conv1(x)))
#         x = self.pool2(F.relu(self.conv2(x)))
#         # !!!
#         x = torch.flatten(x, start_dim=1) # flatten all dimensions except batch
#         # print(x.size())
#         # !!!
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = self.fc3(x)

#         return x

In [ ]:
# import torch.optim as optim

# set_seed(42)

# criterion = nn.CrossEntropyLoss()
# net = Net().to(device)
# optimizer = optim.Adam(net.parameters(), lr=0.001)

# net

/usr/local/lib/python3.11/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): LazyLinear(in_features=0, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=37, bias=True)
)

In [ ]:
# for epoch in range(5):  # loop over the dataset multiple times

#     running_loss = 0.0
#     for data in train_loader:
#         inputs, labels = data
#         inputs, labels = inputs.to(device), labels.to(device)

#         # zero the parameter gradients
#         optimizer.zero_grad()

#         # forward + backward + optimize
#         outputs = net(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         # print statistics
#         running_loss += loss.item()

#     print('[%d/5] loss: %.3f' %
#           (epoch+1 ,  running_loss / 2000))
#     running_loss = 0.0

# print('Finished Training')

[1/5] loss: 2.418
[2/5] loss: 2.108
[3/5] loss: 1.956
[4/5] loss: 1.852
[5/5] loss: 1.766
Finished Training
